In [1]:
import sqlite3
import pandas as pd
import os
import csv

In [2]:

DATABASE = "../../drive_data_v1/v33_koondkorpus_sentences_verb_pattern_obl_20241002-130310_test.db"

OBL_TABLE = "spatial_obl"

TAG = "ELT" # ei tohiks olla vahet kas võtta A/S/ELT, sest lõksud on samad

# how many unique lemmas per verb+comp+case
LEMMA_LIMIT = 8 # limit for number of unique lemmas, maximum number of allowed unique lemmas
    
LINE_DATA_TABLE = f"spatial_obl_{TAG}_n_class"
FILTERED_CLASS_TABLE = f"spatial_obl_n_rare_{LEMMA_LIMIT}lemma"

FNAME = f"n_rare_{LEMMA_LIMIT}lemma"
LARGE_DATA_FILE = f"../data/n_rare/{FNAME}.csv"
LARGE_DATA_FILE_SORTED = f"../data/n_rare/{FNAME}_sorted.csv"


### andmetabelid

In [13]:
conn = sqlite3.connect(DATABASE)
cursor = conn.cursor()

### graafiku punktide info

In [14]:

query = f"""SELECT verb, verb_compound, morph_case, log2_ratio, level, unique_lemmas, ann_unique_lemmas, 
            not_ann_unique_lemmas, olulisus, my_tag, other_tags, annotated, not_annotated, verb_case_count
            FROM {LINE_DATA_TABLE}
            where unique_lemmas <= {LEMMA_LIMIT}
            """

filtered_class = pd.read_sql(query, conn)
filtered_class

,verb,verb_compound,morph_case,log2_ratio,level,unique_lemmas,ann_unique_lemmas,not_ann_unique_lemmas,olulisus,my_tag,other_tags,annotated,not_annotated,verb_case_count
0,aasima,,ad,9.965784,-,2,2.0,4.0,-,2,0,2,6,8
1,abielluma,,abl,9.965784,-,2,2.0,12.0,-,2,0,2,101,103
2,adopteerima,,in,9.965784,-,2,2.0,3.0,-,2,0,2,3,5
3,aduma,,adit,9.965784,-,2,2.0,3.0,-,2,0,2,3,5
4,aeguma,,abl,9.965784,-,2,2.0,7.0,-,2,0,2,8,10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12994,ülistama,,in,2.807355,-,7,6.0,49.0,-,7,1,8,75,83
12995,üllatama,,abl,-0.415037,-,7,3.0,19.0,-,3,4,7,25,32
12996,ümisema,,ad,0.415037,-,7,4.0,8.0,-,4,3,7,10,17
12997,ütlema,üles,el,1.321928,-,7,5.0,13.0,-,5,2,7,15,22


In [15]:
filtered_class["unique_lemmas"].value_counts()

unique_lemmas
2    2767
1    2756
3    2260
4    1661
5    1249
6     944
7     775
8     587
Name: count, dtype: int64

In [16]:
# check that all the verbcomps have no level
filtered_class["level"].value_counts()

level
-    12999
Name: count, dtype: int64

In [6]:
filtered_class["morph_case"].value_counts()

morph_case
ad      2976
in      2595
el      2363
all     1871
adit    1270
abl     1064
ill      860
Name: count, dtype: int64

In [7]:
# adit     9406
# abl      7894
# ill      6290
# these cases din't get 10K sentences, so for these, allow 9 or more lemmas


query = f"""SELECT verb, verb_compound, morph_case, log2_ratio, level, unique_lemmas, ann_unique_lemmas, 
            not_ann_unique_lemmas, olulisus, my_tag, other_tags, annotated, not_annotated, verb_case_count
            FROM {LINE_DATA_TABLE}
            where (unique_lemmas >= 9 and unique_lemmas <= 11 and morph_case IN ('adit') and level == '-' ) 
            OR (unique_lemmas >= 9 and unique_lemmas <= 12 and morph_case IN ('abl') and level == '-' )
            OR (unique_lemmas >= 9 and unique_lemmas <= 15 and morph_case IN ('ill')  and level == '-')
            
            """

added_class = pd.read_sql(query, conn)
added_class




,verb,verb_compound,morph_case,log2_ratio,level,unique_lemmas,ann_unique_lemmas,not_ann_unique_lemmas,olulisus,my_tag,other_tags,annotated,not_annotated,verb_case_count
0,anuma,,abl,-3.700440,-,12,1.0,10.0,-,1,13,14,15,29
1,eelistama,,ill,3.700440,-,12,11.0,13.0,-,13,1,14,15,29
2,eksima,,abl,1.169925,-,12,8.0,32.0,-,9,4,13,137,150
3,emigreerima,,ill,9.965784,-,12,12.0,2.0,-,57,0,57,4,61
4,hindama,,abl,-0.736966,-,12,6.0,48.0,-,6,10,16,84,100
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
366,suutma,,ill,2.906891,-,15,13.0,66.0,-,15,2,17,85,102
367,tõukama,,ill,0.874469,-,15,10.0,40.0,-,11,6,17,46,63
368,töötama,,ill,3.321928,-,15,13.0,20.0,-,20,2,22,24,46
369,viima,üle,ill,9.965784,-,15,15.0,5.0,-,15,0,15,6,21


In [8]:
added_class["morph_case"].value_counts()

morph_case
ill     173
abl      99
adit     99
Name: count, dtype: int64

In [9]:
# check that all the verbcomps have no level
added_class["level"].value_counts()

level
-    371
Name: count, dtype: int64

In [10]:
rare = pd.concat([filtered_class, added_class])

In [11]:
rare

,verb,verb_compound,morph_case,log2_ratio,level,unique_lemmas,ann_unique_lemmas,not_ann_unique_lemmas,olulisus,my_tag,other_tags,annotated,not_annotated,verb_case_count
0,aasima,,ad,9.965784,-,2,2.0,4.0,-,2,0,2,6,8
1,abielluma,,abl,9.965784,-,2,2.0,12.0,-,2,0,2,101,103
2,adopteerima,,in,9.965784,-,2,2.0,3.0,-,2,0,2,3,5
3,aduma,,adit,9.965784,-,2,2.0,3.0,-,2,0,2,3,5
4,aeguma,,abl,9.965784,-,2,2.0,7.0,-,2,0,2,8,10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
366,suutma,,ill,2.906891,-,15,13.0,66.0,-,15,2,17,85,102
367,tõukama,,ill,0.874469,-,15,10.0,40.0,-,11,6,17,46,63
368,töötama,,ill,3.321928,-,15,13.0,20.0,-,20,2,22,24,46
369,viima,üle,ill,9.965784,-,15,15.0,5.0,-,15,0,15,6,21


In [12]:
rare.to_sql(FILTERED_CLASS_TABLE, conn, if_exists="replace", index=False)

13370

In [8]:
query = f"""SELECT * from {FILTERED_CLASS_TABLE}
            """

tbl = pd.read_sql(query, conn)
tbl

,verb,verb_compound,morph_case,log2_ratio,level,unique_lemmas,ann_unique_lemmas,not_ann_unique_lemmas,olulisus,my_tag,other_tags,annotated,not_annotated,verb_case_count
0,aasima,,ad,9.965784,-,2,2.0,4.0,-,2,0,2,6,8
1,abielluma,,abl,9.965784,-,2,2.0,12.0,-,2,0,2,101,103
2,adopteerima,,in,9.965784,-,2,2.0,3.0,-,2,0,2,3,5
3,aduma,,adit,9.965784,-,2,2.0,3.0,-,2,0,2,3,5
4,aeguma,,abl,9.965784,-,2,2.0,7.0,-,2,0,2,8,10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13365,suutma,,ill,2.906891,-,15,13.0,66.0,-,15,2,17,85,102
13366,tõukama,,ill,0.874469,-,15,10.0,40.0,-,11,6,17,46,63
13367,töötama,,ill,3.321928,-,15,13.0,20.0,-,20,2,22,24,46
13368,viima,üle,ill,9.965784,-,15,15.0,5.0,-,15,0,15,6,21


In [9]:
tbl["level"].value_counts()

level
-    13370
Name: count, dtype: int64

In [17]:
tbl["unique_lemmas"].value_counts()

unique_lemmas
2     2767
1     2756
3     2260
4     1661
5     1249
6      944
7      775
8      587
9      120
10      86
11      66
12      49
13      24
15      15
14      11
Name: count, dtype: int64

### võtta spatial_obl tabelist näitelaused koos vajaliku infoga

In [13]:
# lõksud, millel on maksimaalselt x (LIMIT) unikaalset lemmat
# iga lemma jaoks lauseid nii, et kokku oleks u 100K lauset
# proovida käänded ühtlaselt katta, nt 10K lauset iga käände kohta vähemalt

limit_str = ""
if LEMMA_LIMIT != 'max':
    limit_str = f" WHERE rn_group_limit <= 8 "

query = f"""
WITH cleaned AS (
    -- Step 1 & 2: match subset table + remove rows with timex_tag NOT NULL
    SELECT
        d.head_id,
        d.row_loc as head_loc,
        d.form,
        d.lemma,
        d.verb,
        d.verb_compound,
        d.morph_case,
        d.sentence,
        d.sentence_id,
        d.timex_tag,
        d.ekilex_tag,
        d.ner_tag,
        tags
    FROM {OBL_TABLE} AS d
    JOIN {FILTERED_CLASS_TABLE} AS s
      ON d.verb = s.verb
     AND d.verb_compound = s.verb_compound
     AND d.morph_case = s.morph_case
),

distinct_lemmas AS (
    -- Step 3 & 4: for each lemma, pick ONE sentence deterministically
    -- Collects all sentences for (verb, verb_compound, morph_case, lemma), orders by sentence_id, assigns rn_per_lemma
    -- for one (verb, verb_compound, morph_case, lemma) example:
    -- sentence_id 10 → rn_per_lemma = 1
    -- sentence_id 25 → rn_per_lemma = 2
    -- sentence_id 42 → rn_per_lemma = 3 ...
    -- in next part rn_per_lemma = 1 then selects the first row per lemma and therefore guarantees only one lemma per verbcase

    SELECT 
        *,
        ROW_NUMBER() OVER (
            PARTITION BY verb, verb_compound, morph_case, lemma
            ORDER BY sentence_id   -- choose best or earliest sentence
        ) AS rn_per_lemma
    FROM cleaned
),

limited AS (
    -- Step 5: limit to LEMMA_LIMIT unique lemmas per (verb, verb_compound, morph_case)
    -- assigns numbers like:
    -- lemma A → 1
    -- lemma B → 2  ...
    -- lemma ZZZ → 501

    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY verb, verb_compound, morph_case
            ORDER BY lemma        -- choose 500 lexicographically smallest lemmas
        ) AS rn_group_limit
    FROM distinct_lemmas
    WHERE rn_per_lemma ==1      -- keep 1 sentence per lemma
)

-- Step 6: final output
-- rn_group_limit <= 500 cuts off at the 500th lemma. Lemmas that have rn_group_limit 501, 502, ... are not included
SELECT
    sentence_id,
    head_id,
    head_loc, 
    verb,
    verb_compound,
    morph_case,
    lemma,
    form,
    sentence,
    tags,
    timex_tag,
    ekilex_tag,
    ner_tag
FROM limited
WHERE
(morph_case IN ('ad', 'in', 'el', 'all') AND rn_group_limit <= 8)
OR
    (morph_case IN ('adit') AND rn_group_limit <= 11)
OR
    (morph_case IN ('abl') AND rn_group_limit <= 12)
OR
    (morph_case IN ('ill') AND rn_group_limit <= 15)
ORDER BY verb, verb_compound, morph_case, lemma;

"""


In [15]:
spatial_obl_ex = pd.read_sql(query, conn)

In [16]:
spatial_obl_ex

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,tags,timex_tag,ekilex_tag,ner_tag
0,5841232,9373099,11,aasima,,ad,arutelu,arutelul,"Reformierakonna aseesimehe "" äpardust "" aasis ...",,None,None,None
1,1752302,2790222,11,aasima,,ad,hommik,hommikul,"Kas siis kaugele on auto parkimismajja viia , ...",|T|AT|ET|LT|ST|AET|ALT|AST|ELT|LST|,SET,None,None
2,1506454,2397595,11,aasima,,ad,pind,pinnal,"Tsirkuses on labased naljad lubatud , näiteks ...",,None,None,None
3,17384343,26669407,2,aasima,,ad,semester,semestril,Sel semestril umbes pooleks aasid ja beesd .,|T|AT|ET|LT|ST|AET|ALT|AST|ELT|LST|,None,time,None
4,13005812,20809442,1,aasima,,ad,tagasitee,Tagasiteel,Tagasiteel hotelli aasisid kaaslased niigi õnn...,,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...
100746,1148231,1824472,7,šveitsima,,in,laager,laagris,""" Juulikuus olin Prantsusmaal ja šveitsis laag...",,None,None,None
100747,18647429,28204178,16,šveitsima,,in,mägi,mägedes,"Kui nad vele surnud ei ole , siis lauatavad na...",,None,None,None
100748,1027318,1634145,15,šveitsima,,in,politsei,politseis,Matemaatikadoktor Tarmo Soomere istus koos kol...,|L|AL|EL|LS|LT|ALT|ELT|LST|,None,None,ORG
100749,1293678,2055252,15,šveitsima,,in,start,stardis,Aivar Korjus - Jürgen Jakk saavutasid külgvank...,,None,None,None


In [18]:
len(spatial_obl_ex["sentence"].value_counts())

95023

In [19]:
spatial_obl_ex["morph_case"].value_counts()

morph_case
ad      20304
in      18125
el      16570
all     13130
adit    11488
abl     10646
ill     10488
Name: count, dtype: int64

In [20]:
# saving to csv
spatial_obl_ex.to_csv(LARGE_DATA_FILE_SORTED, encoding="utf-8", index = False,sep=",", quoting=csv.QUOTE_MINIMAL)

In [21]:
# saving to database
spatial_obl_ex.to_sql(FNAME, conn, if_exists="replace", index=False)

100751

In [22]:
# shuffle
df = spatial_obl_ex.sample(frac=1)

In [23]:
df.to_csv(LARGE_DATA_FILE, encoding="utf-8", index = False,sep=",", quoting=csv.QUOTE_MINIMAL)

In [18]:
conn.close()

In [3]:
df2 = pd.read_csv(LARGE_DATA_FILE, encoding="utf-8", sep=",")

In [5]:
df2

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,tags,timex_tag,ekilex_tag,ner_tag
0,957330,1524751,4,sõitma,täis,in,tipp,tipus,Tallinnas Kopli poolsaare tipus sõitis neljapä...,NaN,NaN,NaN,NaN
1,12994631,20791012,1,populariseerima,NaN,ad,Inglismaa,Inglismaal,Inglismaal populariseeris jõulupuud alles kuni...,|L|AL|EL|LS|LT|ALT|ELT|LST|,NaN,location,NaN
2,156428,260528,9,kõmama,NaN,el,kinnitus,kinnitusest,20 aastat hiljem kõmas kogu maailm Todor Zhivk...,NaN,NaN,NaN,NaN
3,7127738,11463876,4,veerema,alla,el,luitenõlv,luitenõlvast,( Silinder veereb luitenõlvast alla . ),NaN,NaN,NaN,NaN
4,6136582,9850013,13,saama,sisse,adit,ajakirjandus,ajakirjandusse,""" Lõpetajatest sai enamik kõrgkooli sisse , me...",NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
100746,375878,583597,12,abistama,NaN,adit,Tallinn,Tallinna,"Kuna ei ole ette näha , et ka riik mõnes küsim...",|L|AL|EL|LS|LT|ALT|ELT|LST|,NaN,location,LOC
100747,8976037,14431419,8,ruttama,NaN,abl,meeter,meetrilt,Võtsin palli omaks ja tõstsin umbes kahekümnel...,NaN,NaN,NaN,NaN
100748,11487081,18400010,6,külmuma,NaN,abl,tänav,tänavalt,Tallinna südalinnast leiti eile hommikul tänav...,|L|AL|EL|LS|LT|ALT|ELT|LST|,NaN,location,NaN
100749,3508348,5649502,30,pöörama,välja,el,aken,aknast,""" Olen Ameerikas olnud monster of omniscience ...",NaN,NaN,NaN,NaN


In [20]:
#df2.head()

In [4]:
counts2 = df2.groupby(['verb','verb_compound', 'morph_case'], dropna=False).size().reset_index(name='count').sort_values('count', ascending=False)
counts2

,verb,verb_compound,morph_case,count
862,eraldama,NaN,ill,15
252,andma,edasi,ill,15
2242,jääma,sisse,ill,15
9696,sisaldama,NaN,ill,15
7379,palkama,NaN,ill,15
...,...,...,...,...
7553,peaxima,NaN,in,1
11017,tootma,kokku,ill,1
7556,pedetsema,NaN,ad,1
624,diagnoosima,kokku,in,1
